# H19a: Partial SV Equalization as a One-Step Local Hessian Probe

This notebook is the analysis companion to `run_experiment.py` for
`H19a_PARTIAL_SV_SADDLES`. It **does not duplicate the experiment core**.
Instead, it imports the script's `run_experiment()` function and analyzes the
returned structured results.

## Pair-level identity
- **Script:** executes the toy experiment and exposes reusable structured results.
- **Notebook:** runs the same computation, then presents tables, plots, runtime
  metadata, and a calibrated interpretation.

## Scientific scope of this notebook
The current experiment asks a narrow question:

> After a 50-step warmup in a tiny 2-layer deep linear model, does **one**
> partial-SV-equalized update increase local negative Hessian curvature relative
> to endpoint controls?

This first-pass notebook intentionally keeps that scope small.

### What is measured
- negative Hessian eigenvalue count after one test step
- near-zero Hessian eigenvalue count after one test step
- minimum Hessian eigenvalue after one test step
- loss after one test step
- explicit deltas relative to the warmup point

### What is **not** established here
- not a trajectory study
- not evidence about long-horizon convergence by itself
- not a proof that partial SV equalization "creates saddle points" in general
- not a direct vanilla-SGD comparison in full parameter space

### Control clarification
- `k=0` is a **per-layer norm-matched raw-gradient control**, not plain SGD.
- `k=1` is **degenerate with `k=0`** under the current top-k-to-mean construction.


In [ ]:
from pathlib import Path
import importlib.util
import os
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

print('Notebook analysis environment loaded.')
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'NumPy: {np.__version__}')
print(f'pandas: {pd.__version__}')

## Notebook-safe path discovery and script import

To avoid notebook-only hazards such as reliance on `__file__`, this notebook
locates the repository root from the current working directory (optionally using
`MUON_RG_REPO_ROOT` if set), then imports the experiment module directly from
its file path.


In [ ]:
EXPECTED_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/H19a_PARTIAL_SV_SADDLES/run_experiment.py')

def find_repo_root():
    env_root = os.environ.get('MUON_RG_REPO_ROOT')
    if env_root:
        env_root = Path(env_root).expanduser().resolve()
        if (env_root / EXPECTED_RELATIVE).exists():
            return env_root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / EXPECTED_RELATIVE).exists():
            return candidate

    raise FileNotFoundError(
        'Could not locate repo root containing '
        f'{EXPECTED_RELATIVE}. Run the notebook from inside the repository or '
        'set MUON_RG_REPO_ROOT explicitly.'
    )

repo_root = find_repo_root()
script_path = repo_root / EXPECTED_RELATIVE

spec = importlib.util.spec_from_file_location('h19a_partial_sv_saddles', script_path)
h19a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h19a)

print(f'Repo root:   {repo_root}')
print(f'Script path: {script_path}')
print('Imported run_experiment() and helpers from the script module.')

## Reproducibility, runtime, and configuration

The experiment is deliberately tiny so that the full `32 x 32` Hessian can be
computed by central finite differences at every evaluation point.

The code below runs the shared script entrypoint **as a function** and records
runtime/configuration metadata for the notebook report.


In [ ]:
notebook_wall_start = time.time()
results = h19a.run_experiment(verbose=False)
notebook_wall_elapsed = time.time() - notebook_wall_start

config = results['config']
analysis = results['analysis']
baseline_df = pd.DataFrame(results['baseline_metrics']).sort_values('seed').reset_index(drop=True)
per_k_df = pd.DataFrame(results['per_k_metrics']).sort_values(['seed', 'k']).reset_index(drop=True)
aggregate_df = pd.DataFrame(results['aggregate_by_k']).sort_values('k').reset_index(drop=True)

print('Experiment execution complete.')
print(f"Experiment ID: {results['experiment_id']}")
print(f"Title: {results['title']}")
print(f"Seeds: {config['seeds']}")
print(f"k sweep: {config['k_per_layer']}")
print(f"Warmup steps: {config['warmup_steps']}")
print(f"Hessian finite-difference epsilon: {config['fd_eps']}")
print(f"Script-reported runtime: {analysis['runtime_seconds']:.2f} s")
print(f"Notebook cell wall time: {notebook_wall_elapsed:.2f} s")
print(f"Total Hessian evaluations: {analysis['total_hessian_evaluations']}")
print(f"Hessian-side gradient evaluations: {analysis['total_hessian_gradient_evaluations']}")
print(f"k=0 control note: {config['k0_control']}")
print(f"k=1 note: {config['k1_note']}")
print(f"k=1 == k=0 check: {analysis['k1_matches_k0_within_tolerance']} (max abs diff {analysis['max_abs_step_diff_k1_vs_k0']:.3e})")

## Baseline warmup-point geometry

These are the per-seed metrics at the warmup point **before** the test step.
They define the local geometry from which all `k`-specific one-step probes are
launched.


In [ ]:
baseline_view = baseline_df[
    ['seed', 'initial_loss', 'warmup_loss', 'neg_count', 'near_zero_count', 'min_eig', 'max_eig', 'theta_norm']
].rename(columns={
    'warmup_loss': 'loss_before_step',
    'neg_count': 'neg_before',
    'near_zero_count': 'near_zero_before',
    'min_eig': 'min_eig_before',
    'max_eig': 'max_eig_before',
})

baseline_view.round({
    'initial_loss': 6,
    'loss_before_step': 6,
    'min_eig_before': 6,
    'max_eig_before': 6,
    'theta_norm': 6,
})

## Aggregate per-`k` table with explicit baseline and delta metrics

The table below is the core notebook summary. For each `k`, it reports:
- baseline mean metrics at the warmup point
- mean post-step metrics
- explicit mean deltas relative to the warmup point
- step/gradient alignment diagnostics exposed by the script

This is the minimum serious reporting layer needed to interpret the toy probe
without overstating the result.


In [ ]:
aggregate_view = aggregate_df[[
    'k',
    'k_label',
    'mean_neg_before',
    'mean_neg_after',
    'mean_delta_neg',
    'mean_near_zero_before',
    'mean_near_zero_after',
    'mean_delta_near_zero',
    'mean_min_eig_before',
    'mean_min_eig_after',
    'mean_delta_min',
    'mean_loss_before',
    'mean_loss_after',
    'mean_delta_loss',
    'mean_step_norm',
    'mean_cos_with_raw_grad',
]].rename(columns={
    'k_label': 'control_or_intervention',
    'mean_neg_before': 'neg_before_mean',
    'mean_neg_after': 'neg_after_mean',
    'mean_delta_neg': 'delta_neg_mean',
    'mean_near_zero_before': 'near_zero_before_mean',
    'mean_near_zero_after': 'near_zero_after_mean',
    'mean_delta_near_zero': 'delta_near_zero_mean',
    'mean_min_eig_before': 'min_eig_before_mean',
    'mean_min_eig_after': 'min_eig_after_mean',
    'mean_delta_min': 'delta_min_eig_mean',
    'mean_loss_before': 'loss_before_mean',
    'mean_loss_after': 'loss_after_mean',
    'mean_delta_loss': 'delta_loss_mean',
    'mean_step_norm': 'step_norm_mean',
    'mean_cos_with_raw_grad': 'cos_with_raw_grad_mean',
})

aggregate_view.round(6)

## Per-seed detailed outcomes

The full per-seed table makes it easy to see whether any aggregate trend is
being driven by only one seed or whether the behavior is uniform across the
small sample.


In [ ]:
per_seed_view = per_k_df[[
    'seed',
    'k',
    'k_label',
    'neg_before',
    'neg_after',
    'delta_neg',
    'near_zero_before',
    'near_zero_after',
    'delta_near_zero',
    'min_eig_before',
    'min_eig_after',
    'delta_min',
    'loss_before',
    'loss_after',
    'delta_loss',
    'step_norm',
    'cos_with_raw_grad',
]]

per_seed_view.round(6)

## Plot 1 — negative-eigenvalue count vs `k`

This plot shows seed-level post-step negative-eigenvalue counts together with
their mean. A horizontal line marks the mean baseline count at the warmup point.
If the original mechanistic story were supported here, one would expect an
intermediate-`k` rise above the endpoint controls.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for seed, seed_rows in per_k_df.groupby('seed'):
    ax.plot(seed_rows['k'], seed_rows['neg_after'], marker='o', alpha=0.45, linewidth=1)

ax.plot(aggregate_df['k'], aggregate_df['mean_neg_after'], color='black', marker='o', linewidth=2.5, label='mean post-step count')
ax.axhline(aggregate_df['mean_neg_before'].iloc[0], color='tab:red', linestyle='--', linewidth=1.5, label='mean baseline count')

ax.set_title('Negative Hessian eigenvalue count after one test step')
ax.set_xlabel('k (per-layer equalization depth)')
ax.set_ylabel('negative eigenvalue count')
ax.set_xticks(config['k_per_layer'])
ax.legend()
plt.tight_layout()
plt.show()

## Plot 2 — minimum eigenvalue vs `k`

Because negative-eigenvalue count can saturate, the minimum eigenvalue provides
a complementary measure of whether local negative curvature becomes slightly
more or less severe after the step.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for seed, seed_rows in per_k_df.groupby('seed'):
    ax.plot(seed_rows['k'], seed_rows['min_eig_after'], marker='o', alpha=0.45, linewidth=1)

ax.plot(aggregate_df['k'], aggregate_df['mean_min_eig_after'], color='black', marker='o', linewidth=2.5, label='mean post-step minimum eigenvalue')
ax.axhline(aggregate_df['mean_min_eig_before'].iloc[0], color='tab:red', linestyle='--', linewidth=1.5, label='mean baseline minimum eigenvalue')

ax.set_title('Minimum Hessian eigenvalue after one test step')
ax.set_xlabel('k (per-layer equalization depth)')
ax.set_ylabel('minimum eigenvalue')
ax.set_xticks(config['k_per_layer'])
ax.legend()
plt.tight_layout()
plt.show()

## Plot 3 — loss change vs `k`

This plot is not the main mechanistic test, but it helps interpret whether the
transformed step becomes locally less favorable even when the negative-count
metric itself does not change.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for seed, seed_rows in per_k_df.groupby('seed'):
    ax.plot(seed_rows['k'], seed_rows['delta_loss'], marker='o', alpha=0.45, linewidth=1)

ax.plot(aggregate_df['k'], aggregate_df['mean_delta_loss'], color='black', marker='o', linewidth=2.5, label='mean Δloss')
ax.axhline(0.0, color='tab:red', linestyle='--', linewidth=1.5)

ax.set_title('Loss change after one test step')
ax.set_xlabel('k (per-layer equalization depth)')
ax.set_ylabel('loss_after - loss_before')
ax.set_xticks(config['k_per_layer'])
ax.legend()
plt.tight_layout()
plt.show()

## Calibrated interpretation of the current run

The notebook should state plainly what the present implementation does and does
not support.


In [ ]:
print('Observed result summary')
print('-----------------------')
print(f"Mean baseline negative-eigenvalue count: {analysis['mean_neg_before']:.1f}")
print(f"Mean post-step negative-eigenvalue count at k=0: {analysis['mean_neg_after_k0']:.1f}")
print(f"Mean post-step negative-eigenvalue count at k={config['dim']}: {analysis['mean_neg_after_full']:.1f}")
print(f"Peak mean post-step negative-eigenvalue count: k={analysis['peak_k']} with {analysis['peak_mean_neg_after']:.1f}")
print(f"Intermediate-k increase above endpoints confirmed? {analysis['intermediate_more_neg_confirmed']}")
print(f"k=1 degeneracy check: {analysis['k1_matches_k0_within_tolerance']} (max diff {analysis['max_abs_step_diff_k1_vs_k0']:.3e})")

if analysis['intermediate_more_neg_confirmed']:
    print()
    print('Interpretation: in this default run, an intermediate k does increase the negative-count metric above endpoint controls.')
    print('That would support the specific local-curvature claim for this toy probe, though still only at one-step scope.')
else:
    print()
    print('Interpretation: this default run does NOT show an intermediate-k increase in negative-eigenvalue count.')
    print('The negative-count metric is therefore not evidence here for partial-SV-induced saddle creation.')
    print('Any visible effect in this run is better described through small shifts in minimum eigenvalue and loss, not through extra negative directions.')

## Final conclusion

This first completion pass intentionally keeps the toy experiment unchanged in
spirit while making the reporting honest and reusable. The conclusion below is
computed from the run above and is phrased to match the observed result rather
than the original stronger narrative.


In [ ]:
if analysis['intermediate_more_neg_confirmed']:
    print('Conclusion: within this tiny one-step Hessian probe, intermediate partial equalization does increase negative curvature relative to endpoint controls.')
    print('That would support the local saddle-creation story for this specific setup, while still leaving trajectory-level claims open.')
else:
    print('Conclusion: within this tiny one-step Hessian probe, the current default run does not show an intermediate-k increase in negative Hessian directions by count.')
    print('The strongest observable effect is instead a small change in minimum eigenvalue and loss as k increases.')
    print('Accordingly, this notebook should be read as a careful local-curvature probe, not as a confirmed explanation of why partial equalization underperforms elsewhere.')

print()
print('Recommended next verification after this pass: run the notebook top-to-bottom in a real Jupyter kernel and confirm the tables/plots match the script output exactly.')